# Noise-Type-Aware Image Denoising — Colab runner

**How this notebook works**
- **Code** comes from GitHub (`git clone` / `git pull`). Edit the code in the repository, not in this notebook.
- **Data and pretrained weights** are downloaded to `/content` at the start of every session (~1 min).
- **Results** (CSV, Markdown tables, figures) are written to Google Drive, so they survive when the session ends.

**Before you start:** `Runtime → Change runtime type → T4 GPU`. The GPU only speeds up DnCNN;
everything else also works on the CPU runtime.

Run the cells from top to bottom. Steps 1–4 are needed in every new session.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/erenatasoy00/com0418-noise-aware-denoising.git"
BRANCH = "main"
SAVE_TO_DRIVE = True                              # False: results stay in /content and are lost
DRIVE_RESULTS = "/content/drive/MyDrive/COM0418/results"

## 2. Get the code

For a **private** repository, add a token under 🔑 *Secrets* (left sidebar) named `GITHUB_TOKEN`.

In [ ]:
import os

url = REPO_URL
try:  # private repository: use the token stored in Colab Secrets
    from google.colab import userdata
    url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q -b {BRANCH} {url} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --format='Code version: %h  %s  (%cr)'

## 3. Install packages and set paths

Colab already has PyTorch, OpenCV, scikit-image, Numba, pandas and matplotlib; this adds the rest.

In [ ]:
!pip install -q -r requirements.txt

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime)")

os.environ["DATA_DIR"] = "/content/data"
os.environ["PRETRAINED_DIR"] = "/content/pretrained"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["RESULTS_DIR"] = DRIVE_RESULTS
RESULTS_DIR = os.environ.get("RESULTS_DIR", f"{REPO_DIR}/results")
print("Results ->", RESULTS_DIR)

## 4. Download data and pretrained DnCNN weights

Add `--all` to also get BSD400 (training / cross-validation) and Kodak24 (color).

In [ ]:
!python -m src.download

## 5. Generate noisy images (SO1)

In [ ]:
!python -m src.noise --dataset set12

## 6. Benchmark all methods (SO2, SO3)

Set12 takes about 5 min on a T4 runtime, BSD68 about 30–40 min. BM3D is the slowest method:
add `--skip bm3d` for a quick run.

In [ ]:
!python -m src.benchmark --dataset set12

## 7. Figures

In [ ]:
!python -m src.plots --dataset set12

## 8. DnCNN sanity check (should match the published 31.73 / 29.23 / 26.23 dB on BSD68)

In [ ]:
!python -m src.dncnn

## 9. Noise-type classifier and noise-level estimator (SO4)

In [ ]:
!python -m src.noise_analysis

## 10. Show the results

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

results = Path(RESULTS_DIR)
for report in ["set12_summary.md", "dncnn_sanity_check.md", "noise_classifier.md"]:
    if (results / report).exists():
        display(Markdown((results / report).read_text()))
for figure in sorted((results / "figures").glob("*.png")):
    print(figure.name)
    display(Image(str(figure), width=900))

## Notes

- **Runtime numbers:** Colab's CPU (2 vCPUs) is slower than a laptop, and DnCNN runs on the GPU here.
  The proposal's targets (< 1 s classical, < 3 s CNN) are for a laptop CPU, so take the final runtime
  table on a laptop with `DENOISE_DEVICE=cpu python -m src.benchmark ...`. The benchmark CSV records
  the device for every DnCNN row.
- **Changing code:** edit and push in the repository, then re-run step 2 (`git pull`) here.
- **Session ended?** Re-run steps 1–4; earlier results are still in Drive.